# RSNA — DenseNet-121, split por paciente, Grad-CAM y análisis de sesgo

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental,
> no validado clínicamente.

Este notebook se lanza por API (`kaggle kernels push`) y corre entero en la GPU
de Kaggle. Los datos ya están montados en `/kaggle/input`, así que no se descarga
nada. Todo lo que se guarde en `/kaggle/working` vuelve como salida del kernel.

Código: https://github.com/GGGuardin/chest-xray-pneumonia

In [ ]:
import subprocess, sys, os, time
T0 = time.time()

# El repositorio se clona fuera de /kaggle/working para no inflar la salida
subprocess.run(['rm', '-rf', '/tmp/repo'], check=False)
subprocess.run(['git', 'clone', '--depth', '1', '-q',
                'https://github.com/GGGuardin/chest-xray-pneumonia.git', '/tmp/repo'], check=True)
os.chdir('/tmp/repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'pydicom'], check=True)

import torch
print('torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE')
print('entradas montadas:', os.listdir('/kaggle/input'))

## 1. Manifiesto y split POR PACIENTE

Lee además sexo, edad y proyección AP/PA de las cabeceras DICOM: sin esos
metadatos no hay análisis de subgrupos.

In [ ]:
RSNA = '/kaggle/input/rsna-pneumonia-detection-challenge'
OUT = '/kaggle/working'

!python -m src.prepare_data --dataset rsna --root {RSNA} --out {OUT}/manifest_rsna.csv

In [ ]:
import pandas as pd
from src.data import split_summary

df = pd.read_csv(f'{OUT}/manifest_rsna.csv')
print(split_summary(df).to_string())
print('\nProyeccion:\n', df['view'].value_counts().to_string())
print('\nSexo:\n', df['sex'].value_counts().to_string())
print('\nEdad: mediana %.0f, rango %.0f-%.0f' % (df.age.median(), df.age.min(), df.age.max()))

# Comprobacion explicita de la propiedad critica del proyecto
assert (df.groupby('patient_id')['split'].nunique() == 1).all(), 'FUGA DE DATOS entre splits'
print('\nOK: ningun paciente aparece en mas de un split.')

## 2. Entrenamiento

DenseNet-121 preentrenada en ImageNet, 224x224, `pos_weight` para el desbalance,
mixed precision. Se guarda checkpoint en cada mejora de AUROC de validación.

In [ ]:
!python -m src.train --config configs/rsna.yaml \
    --manifest {OUT}/manifest_rsna.csv \
    --out-dir {OUT}/runs/rsna_densenet121 \
    --batch-size 64

In [ ]:
import pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs(f'{OUT}/reports', exist_ok=True)
h = pd.read_csv(f'{OUT}/runs/rsna_densenet121/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='val')
ax[0].set_title('loss'); ax[0].set_xlabel('epoca'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h.epoch, h.train_auroc, label='train'); ax[1].plot(h.epoch, h.val_auroc, label='val')
ax[1].set_title('AUROC'); ax[1].set_xlabel('epoca'); ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(f'{OUT}/reports/curvas_entrenamiento.png', dpi=150)
print(h.to_string(index=False))

## 3. Evaluación en test interno (split por paciente)

In [ ]:
!python -m src.evaluate --checkpoint {OUT}/runs/rsna_densenet121/best.pth \
    --manifest {OUT}/manifest_rsna.csv --split test --out-dir {OUT}/reports/rsna_test

## 4. Grad-CAM y auditoría de shortcut learning

Además de los mapas, se calcula qué fracción de la energía del CAM cae en el
marco exterior de la imagen: si es alta, el modelo mira bordes y marcadores en
vez del parénquima pulmonar.

In [ ]:
!python -m src.explain --checkpoint {OUT}/runs/rsna_densenet121/best.pth \
    --manifest {OUT}/manifest_rsna.csv --split test --n 16 --out-dir {OUT}/reports/gradcam

## 5. Análisis de sesgo: infradiagnóstico por subgrupos

FNR por sexo, grupo de edad, proyección AP/PA y sus intersecciones.

In [ ]:
!python -m src.fairness --predictions {OUT}/reports/rsna_test/predictions.csv \
    --out-dir {OUT}/reports/fairness

## 6. Validación externa

El modelo entrenado en RSNA (adultos, urgencias de EE. UU.) se evalúa sobre
"Chest X-Ray Images (Pneumonia)" (pediátrico, un solo hospital de Guangzhou).
Criterio del proyecto: una caída de AUROC > 0,10 es un problema de
generalización que hay que documentar y analizar, no esconder.

In [ ]:
EXT = '/kaggle/input/chest-xray-pneumonia'
import os
if os.path.exists(EXT):
    !python -m src.prepare_data --dataset kaggle_pneumonia --root {EXT} --out {OUT}/manifest_externo.csv
    !python -m src.evaluate --checkpoint {OUT}/runs/rsna_densenet121/best.pth \
        --manifest {OUT}/manifest_externo.csv --split all --out-dir {OUT}/reports/externo_kaggle
else:
    print('Dataset externo no montado; se omite la validacion externa.')

## 7. Resumen

In [ ]:
import json, glob, time

resumen = {}
for nombre, ruta in [('test_interno_rsna', f'{OUT}/reports/rsna_test/metrics.json'),
                     ('externo_kaggle', f'{OUT}/reports/externo_kaggle/metrics.json')]:
    if os.path.exists(ruta):
        with open(ruta) as f:
            resumen[nombre] = json.load(f)

for ruta in [f'{OUT}/reports/fairness/fairness.json', f'{OUT}/reports/gradcam/shortcut_audit.json']:
    if os.path.exists(ruta):
        with open(ruta) as f:
            d = json.load(f)
        resumen[os.path.basename(os.path.dirname(ruta))] = (
            {k: v for k, v in d.items() if k != 'detalle'})

interno = resumen.get('test_interno_rsna', {}).get('auroc')
externo = resumen.get('externo_kaggle', {}).get('auroc')
if interno and externo:
    resumen['caida_auroc_externa'] = round(interno - externo, 4)
resumen['minutos_totales'] = round((time.time() - T0) / 60, 1)

with open(f'{OUT}/resumen.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

print('\nArchivos de salida:')
for p in sorted(glob.glob(f'{OUT}/**/*', recursive=True)):
    if os.path.isfile(p):
        print(f'  {os.path.getsize(p)/1e6:8.2f} MB  {p}')